### MOVIE RECOMMENDATION

## This project is a simple content-based movie recommendation system that uses genre similarity to recommend movies
## It is built using Python and pandas, and computes cosine similarity between one-hot encoded genre vectors to find movies similar to a user's input.

## 📁 Workflow Summary

### 1. Load Dataset
- Import a local CSV file of movies that includes genre labels.

### 2. Preprocessing
- Replace any occurrences of `'(no genres listed)'` with an empty string.
- Split genre strings (e.g., "Action|Adventure") into lists.
- One-hot encode the genres into binary vectors for each genre.

### 3. Build Similarity Matrix
- Use cosine similarity to compare one-hot encoded genre vectors.
- Create a similarity matrix where each movie is compared with all others.

### 4. User Input
- Accept a movie title from the user (year is optional).
- Clean the input by removing any year suffix.
- Match the cleaned title against entries in the dataset.

### 5. Recommendations
- Find similar movies using the similarity matrix.
- Filter by a similarity threshold.
- Return the top N most similar movies based on genre.


### STEP 1

In [9]:
import pandas as pd

movies = pd.read_csv(r'C:\Users\HP\Desktop\movie data\ml-latest-small\movies.csv')  # contains movieId, title, genres
ratings = pd.read_csv(r"C:\Users\HP\Desktop\movie data\ml-latest-small\ratings.csv")  # contains userId, movieId, rating, timestamp

print(movies.head())
print(ratings.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


In [11]:
import pandas as pd

# Import dataset (CSV file as an example)
df = pd.read_csv(r"C:\Users\HP\Desktop\movie data\ml-latest-small\movies.csv")

# Display the first 5 rows in a clean table
df.head()


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [12]:
import pandas as pd

# Import dataset (CSV file as an example)
df = pd.read_csv(r'C:\Users\HP\Desktop\movie data\ml-latest-small\movies.csv')

# Display the first 5 rows in a clean table
df.head()


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


### STEP 2

In [19]:
# Replace '(no genres listed)' with empty string for cleaner processing
movies['genres'] = movies['genres'].replace('(no genres listed)', '')

# One-hot encode genres directly from the string (no need to split manually)
genres_encoded = movies['genres'].str.get_dummies(sep='|')

# Concatenate the original dataframe with the one-hot encoded genres
movies_encoded = pd.concat([movies[['movieId', 'title']], genres_encoded], axis=1)

# View the result
movies_encoded()


,movieId,title,[''],"['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy']","['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'IMAX']","['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Romance']","['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Sci-Fi', 'IMAX']","['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Sci-Fi']","['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Western']","['Action', 'Adventure', 'Animation', 'Children', 'Comedy']",...,"['Romance', 'War']","['Romance', 'Western']",['Romance'],"['Sci-Fi', 'IMAX']","['Sci-Fi', 'Thriller', 'IMAX']","['Sci-Fi', 'Thriller']",['Sci-Fi'],['Thriller'],['War'],['Western']
0,1,Toy Story (1995),0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,Waiting to Exhale (1995),0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Father of the Bride Part II (1995),0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Step 3

In [20]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity on genre vectors only (drop movieId and title)
genre_features = movies_encoded.drop(['movieId', 'title'], axis=1)

# Compute the cosine similarity matrix
cosine_sim = cosine_similarity(genre_features)

# Optional: convert to DataFrame for easy inspection
cosine_sim_df = pd.DataFrame(cosine_sim, index=movies_encoded['title'], columns=movies_encoded['title'])

# View a slice of the similarity matrix
cosine_sim_df.iloc[:5, :5]


title,Toy Story (1995),Jumanji (1995),Grumpier Old Men (1995),Waiting to Exhale (1995),Father of the Bride Part II (1995)
title,,,,,
Toy Story (1995),1.0,0.0,0.0,0.0,0.0
Jumanji (1995),0.0,1.0,0.0,0.0,0.0
Grumpier Old Men (1995),0.0,0.0,1.0,0.0,0.0
Waiting to Exhale (1995),0.0,0.0,0.0,1.0,0.0
Father of the Bride Part II (1995),0.0,0.0,0.0,0.0,1.0


### Step 4

In [22]:
# Choose a reference movie
movie_name = "Jumanji (1995)"

# Get similarity scores for that movie
similarities = cosine_sim_df[movie_name]

# Filter movies with similarity >= 0.8 (and not the movie itself)
similar_movies = similarities[(similarities >= 0.8) & (similarities.index != movie_name)]

# Sort in descending order
similar_movies = similar_movies.sort_values(ascending=False)

# Show results
similar_movies


title
Indian in the Cupboard, The (1995)                                                                1.0
Water Horse: Legend of the Deep, The (2007)                                                       1.0
Pete's Dragon (2016)                                                                              1.0
Alice Through the Looking Glass (2016)                                                            1.0
Pan (2015)                                                                                        1.0
The Cave of the Golden Rose (1991)                                                                1.0
Seventh Son (2014)                                                                                1.0
Percy Jackson: Sea of Monsters (2013)                                                             1.0
Chronicles of Narnia: The Voyage of the Dawn Treader, The (2010)                                  1.0
Alice in Wonderland (1933)                                                  

### advance search 

In [41]:
import pandas as pd
import re
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

# Assuming 'movies_encoded' and 'cosine_sim_df' are already set up

# Function to remove year from the movie title
def clean_movie_name(title):
    # Remove year (like (1995), (2001), etc.)
    return re.sub(r"\(\d{4}\)", "", title).strip()

# Function to find the closest match for a movie (ignores the year)
def find_closest_movie(user_input):
    # Clean the user input
    cleaned_input = clean_movie_name(user_input)
    
    # Check for movies in the dataset by matching the cleaned input
    matches = [movie for movie in movies_encoded['title'] if clean_movie_name(movie) == cleaned_input]
    
    if len(matches) == 1:
        return matches[0]  # Exact match
    elif len(matches) > 1:
        return matches  # Multiple matches (can decide how to handle this)
    else:
        return None  # No match found

# Function to recommend movies based on a title
def recommend_movies(title, threshold=0.8, top_n=5):
    """
    Recommends similar movies based on cosine similarity of genres.

    Parameters:
    - title (str): The movie title to base recommendations on.
    - threshold (float): The minimum similarity score for recommendations (default is 0.8).
    - top_n (int): The number of top similar movies to return (default is 5).
    
    Returns:
    - Pandas Series: A list of similar movies sorted by similarity score.
    """
    # Find closest match in the dataset (ignoring the year)
    closest_match = find_closest_movie(title)
    
    if closest_match is None:
        return f"'{title}' not found in movie list."
    
    # Get the similarity scores for the matched movie
    similarities = cosine_sim_df[closest_match]
    
    # Filter out movies with similarity >= threshold and not the movie itself
    similar_movies = similarities[(similarities >= threshold) & (similarities.index != closest_match)]
    
    # Sort by similarity score in descending order and get top_n results
    return similar_movies.sort_values(ascending=False).head(top_n)

# Test the function
movie_title = "Toy Story"  # Without the year
print(f"Recommendations for '{movie_title}':")
recommendations = recommend_movies(movie_title, threshold=0.8, top_n=5)

# Show the results in a table form if using a Jupyter notebook
display(recommendations)


Recommendations for 'Toy Story':


title
Antz (1998)                                       1.0
Toy Story 2 (1999)                                1.0
Adventures of Rocky and Bullwinkle, The (2000)    1.0
Emperor's New Groove, The (2000)                  1.0
Monsters, Inc. (2001)                             1.0
Name: Toy Story (1995), dtype: float64